In [1]:
import pandas as pd
df=pd.read_csv('cleaned_data.csv')

print(df.isnull().sum())
print(df.duplicated().sum())
df.sample(10)




answers    0
label      0
dtype: int64
0


,answers,label
29595,"[""The capacity of a magazine, which is the par...",1
2365,['I hope this does n\'t double - comment . I s...,0
10689,"[""You body uses most of your calories just to ...",0
3332,['Automobiles tend to always be a few years be...,0
23500,['The Keystone Pipeline is a proposed pipeline...,1
33801,"[""The internet is able to reach overseas becau...",1
38997,"[""A nautical mile is a unit of distance used i...",1
1786,['I \'ll try to actually explain it simple ter...,0
16069,['The margarita is a Mexican cocktail consisti...,0
26035,['Birth control pills are a very effective for...,1


In [30]:
import ast

def clean_text(x):
    if isinstance(x, list):
        return " ".join(map(str, x))

    if isinstance(x, dict):   # 🔥 ADD THIS
        return " ".join(map(str, x.values()))

    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)

            if isinstance(parsed, list):
                return " ".join(map(str, parsed))
            else:
                return str(parsed)

        except:
            return x

    return str(x)

In [3]:
df['answers'] = df['answers'].str.lower()
df['answers']=df['answers'].apply(clean_text)

df.head()

,answers,label
0,"basically there are many categories of "" best ...",0
1,salt is good for not dying in car crashes and ...,0
2,the way it works is that old tv stations got a...,0
3,you ca n't just go around assassinating the le...,0
4,wanting to kill the shit out of germans drives...,0


In [4]:
df_feat = df.copy()
df_feat.sample(5)

,answers,label
37354,yawning is a natural reflex that most people e...,1
45018,i'm sorry to hear that you are experiencing pa...,1
34051,eurovision is a big singing competition that h...,1
23178,people like alcohol for a variety of reasons. ...,1
11726,why would n't it be legal ? it 's an organizat...,0


In [5]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
nlp.add_pipe("sentencizer")

In [6]:
def load_word_list(path):
    with open(path, "r", encoding="utf-8") as f:
        return set(line.strip().lower() for line in f if line.strip())

chat_words = load_word_list("chat_words.txt")
function_words = load_word_list("function_words.txt")
discourse_markers = load_word_list("discourse_markers.txt")

In [7]:
def split_sentences(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

In [8]:
def extract_features(text):
    doc = nlp(text)

    tokens = [t.text.lower() for t in doc if not t.is_space]
    alpha_tokens = [t.text.lower() for t in doc if t.is_alpha]

    total_tokens = len(tokens)
    total_alpha = len(alpha_tokens)

    if total_tokens == 0 or total_alpha == 0:
        return {
            "chat_word_count": 0,
            "punct_count": 0,
            "function_count": 0,
            "discourse_count": 0,
            # spelling_error_ratio removed: is_oov flags proper nouns/technical
            # terms as errors, introducing more noise than signal
            "ttr": 0,
            "function_word_ratio": 0,
            "discourse_ratio": 0,
            "sentence_length": 0,
            "avg_word_length": 0,
        }

    chat_word_count = sum(1 for t in tokens if t in chat_words)
    punct_count = sum(1 for t in doc if t.is_punct)

    function_count = sum(1 for t in tokens if t in function_words)
    discourse_count = sum(1 for t in tokens if t in discourse_markers)

    return {
        "chat_word_count": chat_word_count,
        "punct_count": punct_count,
        "function_count": function_count,
        "discourse_count": discourse_count,
        # spelling_error_ratio removed: spaCy is_oov misclassifies domain
        # terms, proper nouns, and abbreviations as spelling errors,
        # producing near-constant noise (ratio ~1.0 across all samples)
        "ttr": len(set(alpha_tokens)) / total_alpha,
        "function_word_ratio": function_count / total_tokens,
        "discourse_ratio": discourse_count / total_tokens,
        "sentence_length": total_tokens,
        "avg_word_length": sum(len(t) for t in alpha_tokens) / total_alpha,
    }

In [9]:
def aggregate_sentence_features(sentences):
    feats = [extract_features(s) for s in sentences]

    if len(feats) == 0:
        return {}

    df = pd.DataFrame(feats)

    return {
        "chat_word_count": df["chat_word_count"].sum(),
        "punct_count": df["punct_count"].sum(),

        "function_count": df["function_count"].sum(),
        "discourse_count": df["discourse_count"].sum(),

        # spelling_error_ratio removed — see extract_features for rationale
        "ttr": df["ttr"].mean(),
        "function_word_ratio": df["function_word_ratio"].mean(),
        "discourse_ratio": df["discourse_ratio"].mean(),

        "sentence_length_mean": df["sentence_length"].mean(),
        # fillna(0): single-sentence docs produce NaN std; 0 is the correct
        # value (no variance when there is only one sentence)
        "sentence_length_std": df["sentence_length"].std(ddof=1) if len(df) > 1 else 0.0,

        "avg_word_length": df["avg_word_length"].mean(),
    }

In [10]:
def build_final_features(df):
    rows = []

    for _, row in df.iterrows():
        text = row["answers"]
        label = row["label"]

        # 1. split into sentences
        sentences = split_sentences(text)

        # 2. aggregate sentence-level features → document-level features
        features = aggregate_sentence_features(sentences)

        # 3. attach label
        features["label"] = label

        rows.append(features)

    return pd.DataFrame(rows)

In [11]:
#df_final = build_final_features(df_feat)

In [12]:
#import joblib
#joblib.dump(df_final, "final_features.pkl")

In [13]:
import joblib
df_feat = joblib.load("final_features.pkl")

In [14]:
df_feat.head()

,chat_word_count,punct_count,function_count,discourse_count,spelling_error_ratio,ttr,function_word_ratio,discourse_ratio,sentence_length_mean,sentence_length_std,avg_word_length,label
0,2.0,48.0,118.0,15.0,1.0,0.899103,0.442874,0.059134,24.272727,14.185139,4.373566,0
1,1.0,46.0,192.0,24.0,1.0,0.918157,0.458618,0.049649,20.750000,10.289877,4.537067,0
2,4.0,55.0,196.0,24.0,1.0,0.900015,0.391254,0.042689,36.923077,23.514044,4.643522,0
3,1.0,20.0,92.0,12.0,1.0,0.933176,0.415362,0.055521,22.111111,19.303137,4.923874,0
4,1.0,29.0,147.0,15.0,1.0,0.868949,0.512040,0.050100,31.666667,12.549900,4.692924,0


In [15]:
df_final = pd.concat(
    [
        df[["answers", "label"]].reset_index(drop=True),
        df_feat.drop(columns=["label"], errors="ignore").reset_index(drop=True)
    ],
    axis=1
)
df_final.head()

,answers,label,chat_word_count,punct_count,function_count,discourse_count,spelling_error_ratio,ttr,function_word_ratio,discourse_ratio,sentence_length_mean,sentence_length_std,avg_word_length
0,"basically there are many categories of "" best ...",0,2.0,48.0,118.0,15.0,1.0,0.899103,0.442874,0.059134,24.272727,14.185139,4.373566
1,salt is good for not dying in car crashes and ...,0,1.0,46.0,192.0,24.0,1.0,0.918157,0.458618,0.049649,20.750000,10.289877,4.537067
2,the way it works is that old tv stations got a...,0,4.0,55.0,196.0,24.0,1.0,0.900015,0.391254,0.042689,36.923077,23.514044,4.643522
3,you ca n't just go around assassinating the le...,0,1.0,20.0,92.0,12.0,1.0,0.933176,0.415362,0.055521,22.111111,19.303137,4.923874
4,wanting to kill the shit out of germans drives...,0,1.0,29.0,147.0,15.0,1.0,0.868949,0.512040,0.050100,31.666667,12.549900,4.692924


In [16]:
df_feat.head()

,chat_word_count,punct_count,function_count,discourse_count,spelling_error_ratio,ttr,function_word_ratio,discourse_ratio,sentence_length_mean,sentence_length_std,avg_word_length,label
0,2.0,48.0,118.0,15.0,1.0,0.899103,0.442874,0.059134,24.272727,14.185139,4.373566,0
1,1.0,46.0,192.0,24.0,1.0,0.918157,0.458618,0.049649,20.750000,10.289877,4.537067,0
2,4.0,55.0,196.0,24.0,1.0,0.900015,0.391254,0.042689,36.923077,23.514044,4.643522,0
3,1.0,20.0,92.0,12.0,1.0,0.933176,0.415362,0.055521,22.111111,19.303137,4.923874,0
4,1.0,29.0,147.0,15.0,1.0,0.868949,0.512040,0.050100,31.666667,12.549900,4.692924,0


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score

In [18]:
X_ans = df_final["answers"]
y = df_final["label"]
X_custom = df_final.drop(columns=["answers", "label"])

In [19]:
X_ans_train, X_ans_test, X_custom_train, X_custom_test, y_train, y_test = train_test_split(
    X_ans,
    X_custom,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Maintain class balance
)

In [20]:

import numpy as np

print(X_custom_train.isna().sum())

chat_word_count            2
punct_count                2
function_count             2
discourse_count            2
spelling_error_ratio       2
ttr                        2
function_word_ratio        2
discourse_ratio            2
sentence_length_mean       2
sentence_length_std     1213
avg_word_length            2
dtype: int64


In [21]:
X_custom_train = X_custom_train.fillna(0)
X_custom_test= X_custom_test.fillna(0)

In [22]:
tfidf = TfidfVectorizer(
    max_features=10_000,
    ngram_range=(1, 2),
    min_df=3,
    sublinear_tf=True
)
X_train_tfidf = tfidf.fit_transform(X_ans_train)
X_test_tfidf = tfidf.transform(X_ans_test)


In [23]:
feature_cols = X_custom_train.columns.tolist()
joblib.dump(feature_cols, "feature_columns.pkl")
scaler = StandardScaler(with_mean=False) 
X_train_custom_scaled = scaler.fit_transform(X_custom_train)
X_test_custom_scaled = scaler.transform(X_custom_test)

In [24]:
X_train = hstack([X_train_tfidf, X_train_custom_scaled])
X_test = hstack([X_test_tfidf, X_test_custom_scaled])


In [26]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # Handle class imbalance if needed
    random_state=42
)
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [27]:
y_pred = model.predict(X_test)

# Evaluate model
print("Classification Report on Test Set:")
print(classification_report(y_test, y_pred, target_names=['Human', 'AI']))
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

print("Confusion Matrix on Test Set:")
print(confusion_matrix(y_test, y_pred))

Classification Report on Test Set:
              precision    recall  f1-score   support

       Human       0.98      0.98      0.98      4489
          AI       0.98      0.98      0.98      4661

    accuracy                           0.98      9150
   macro avg       0.98      0.98      0.98      9150
weighted avg       0.98      0.98      0.98      9150

Accuracy: 0.9807
Confusion Matrix on Test Set:
[[4395   94]
 [  83 4578]]


In [28]:
joblib.dump(model, 'logistic_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(scaler, 'feature_scaler.pkl')


['feature_scaler.pkl']

In [35]:
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import hstack

# Load saved artifacts
model = joblib.load('logistic_model.pkl')
tfidf = joblib.load('tfidf_vectorizer.pkl')
scaler = joblib.load('feature_scaler.pkl')
feature_cols = joblib.load('feature_columns.pkl')  # saved column order from training

# Custom feature extraction functions must be defined above:
# split_sentences, clean_text, extract_features, aggregate_sentence_features

# Input text
text = """In this pipeline, I am performing data cleaning before passing features into the model. First, I replace all missing values (NaN) with 0 to ensure there are no undefined values in the dataset. Then, I handle infinite values (positive and negative infinity) by replacing them with 0, since such values can occur due to division operations in feature engineering. Finally, I align the feature DataFrame with the exact column order used during training by reindexing using the saved feature list. This ensures that the model receives input in the same structure it was trained on, which is critical for correct predictions and avoiding shape or alignment errors."""

# Step 1: Split text into sentences
sentences = split_sentences(text)

# Step 2: Create chunks (same granularity used conceptually during training)
chunk_size = 3  # Adjust as needed
chunks = [' '.join(sentences[i:i+chunk_size]) for i in range(0, len(sentences), chunk_size)]

# Step 3: Extract features for each chunk and aggregate
chunk_features = []
for chunk in chunks:
   sentences_in_chunk = [clean_text(s) for s in split_sentences(chunk)]
   aggregated_features = aggregate_sentence_features(sentences_in_chunk)
   chunk_features.append(aggregated_features)

# Step 4: Convert chunk features to DataFrame and sanitize
features_df = pd.DataFrame(chunk_features)
features_df = features_df.replace([np.inf, -np.inf], 0)  # guard division artefacts
features_df = features_df.fillna(0)                       # guard single-sentence std NaN

# Step 5: Align column order with training (done once — duplicate removed)
features_df = features_df.reindex(columns=feature_cols, fill_value=0)

# Step 6: Scale custom features
custom_scaled = scaler.transform(features_df)

# Step 7: Transform each chunk using TF-IDF
chunk_tfidf = tfidf.transform(chunks)

# Step 8: Combine TF-IDF and custom features
X_chunk = hstack([chunk_tfidf, custom_scaled])

# Step 9: Predict probabilities for each chunk
chunk_probs = model.predict_proba(X_chunk)[:, 1]  # probability of class 1 (AI)

# Step 10: Aggregate chunk probabilities (average)
final_score = np.mean(chunk_probs)

# Step 11: Final prediction using threshold
final_prediction = 1 if final_score > 0.5 else 0

print("Final aggregated score:", final_score)
print("Final prediction:", "AI" if final_prediction == 1 else "Human")
